In [1]:
# ============================================================
# MULTI-HORIZON EXPLAINABLE BOOSTING MACHINE (EBM)
# Pipeline Phase 5 | KBS Paper | Coastal Wave Height Forecasting
# Author: Research Pipeline | Target Journal: Q1
# ============================================================

# ── 0. ENVIRONMENT SETUP ────────────────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

pip_install("interpret")
pip_install("scikit-learn")

import os
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show

warnings.filterwarnings("ignore")
sns.set_style("ticks")

print("=" * 65)
print("   PHASE 5 — EBM MULTI-HORIZON TRAINING PIPELINE")
print("=" * 65)

# ── 1. PATH CONFIGURATION ────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

INPUT_DIR  = "/content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/"
OUTPUT_DIR = "/content/drive/MyDrive/KBS_Paper/Outputs/5_EBM_KBS/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n[✓] Output directory ready: {OUTPUT_DIR}")

# ── 2. LOAD DATA ─────────────────────────────────────────────
print("\n[1/6] Loading feature-engineered datasets...")

train_raw = pd.read_csv(os.path.join(INPUT_DIR, "X_train_KBS.csv"), parse_dates=["time"])
oos_raw   = pd.read_csv(os.path.join(INPUT_DIR, "X_oos_KBS.csv"),   parse_dates=["time"])

train_raw = train_raw.sort_values("time").set_index("time")
oos_raw   = oos_raw.sort_values("time").set_index("time")

assert "target_buoy_hs" in train_raw.columns, "'target_buoy_hs' column not found in training data!"
assert "target_buoy_hs" in oos_raw.columns,   "'target_buoy_hs' column not found in OOS data!"

print(f"    Train shape : {train_raw.shape}")
print(f"    OOS   shape : {oos_raw.shape}")
print(f"    Train period: {train_raw.index.min()} → {train_raw.index.max()}")
print(f"    OOS   period: {oos_raw.index.min()} → {oos_raw.index.max()}")

# ── 3. GLOBAL CONFIG ─────────────────────────────────────────
HORIZONS     = [3, 6, 12, 24]   # Forecast horizons in hours
TIME_RES_H   = 3                 # Data resolution: 3-hourly
TOP_K        = 100               # SelectKBest: top features for EBM
TARGET_COL   = "target_buoy_hs"

EBM_PARAMS = dict(
    interactions = 10,
    max_bins     = 128,
    outer_bags   = 4,
    inner_bags   = 0,
    random_state = 42,
)

# ── 4. METRIC HELPER ─────────────────────────────────────────
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Returns RMSE, MAE, R2, MAPE."""
    mask = y_true != 0  # Avoid division by zero in MAPE
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE" : mean_absolute_error(y_true, y_pred),
        "R2"  : r2_score(y_true, y_pred),
        "MAPE": mape,
    }

# ── 5. MULTI-HORIZON TRAINING LOOP ───────────────────────────
print("\n[2/6] Starting multi-horizon EBM training loop...")
print("-" * 65)

all_metrics       = []
train_pred_frames = []
oos_pred_frames   = []
trained_models    = {}     # horizon_h → fitted EBM

for h in HORIZONS:
    step_shift = int(h / TIME_RES_H)
    print(f"\n  ┌─ Horizon: +{h}h  (step_shift = {step_shift} × 3h)")

    # ── 5a. Dynamic Target Creation ──────────────────────────
    # Shift target backward so row[t] carries Hs[t + step_shift]
    train_h = train_raw.copy()
    oos_h   = oos_raw.copy()

    train_h["__target__"] = train_h[TARGET_COL].shift(-step_shift)
    oos_h["__target__"]   = oos_h[TARGET_COL].shift(-step_shift)

    # Drop NaN rows introduced by shift
    train_h = train_h.dropna(subset=["__target__"])
    oos_h   = oos_h.dropna(subset=["__target__"])

    # Separate targets and features (drop original target column)
    y_train = train_h["__target__"].values
    y_oos   = oos_h["__target__"].values

    feat_cols = [c for c in train_h.columns if c not in [TARGET_COL, "__target__"]]
    X_train_full = train_h[feat_cols].values
    X_oos_full   = oos_h[feat_cols].values

    idx_train = train_h.index
    idx_oos   = oos_h.index

    print(f"  │  Train samples: {len(y_train):,}  |  OOS samples: {len(y_oos):,}")
    print(f"  │  Feature pool : {X_train_full.shape[1]} features")

    # ── 5b. Memory-Safe Feature Selection (SelectKBest) ──────
    k_actual = min(TOP_K, X_train_full.shape[1])
    print(f"  │  Running SelectKBest (k={k_actual}) to reduce RAM footprint...")

    selector = SelectKBest(score_func=f_regression, k=k_actual)
    X_train_sel = selector.fit_transform(X_train_full, y_train)
    X_oos_sel   = selector.transform(X_oos_full)

    selected_mask  = selector.get_support()
    selected_feats = [feat_cols[i] for i in np.where(selected_mask)[0]]
    print(f"  │  Selected features: {X_train_sel.shape[1]}")

    # ── 5c. EBM Initialization & Training ────────────────────
    print(f"  │  Initialising ExplainableBoostingRegressor...")
    ebm = ExplainableBoostingRegressor(**EBM_PARAMS)

    print(f"  │  Fitting EBM on {X_train_sel.shape[0]:,} training samples...")
    ebm.fit(X_train_sel, y_train)
    trained_models[h] = {"model": ebm, "selector": selector, "features": selected_feats}
    print(f"  │  Training complete.")

    # ── 5d. Predictions ──────────────────────────────────────
    pred_train = ebm.predict(X_train_sel)
    pred_oos   = ebm.predict(X_oos_sel)

    # ── 5e. Metrics ──────────────────────────────────────────
    m_train = compute_metrics(y_train, pred_train)
    m_oos   = compute_metrics(y_oos,   pred_oos)

    print(f"  │  Train → RMSE={m_train['RMSE']:.4f} | MAE={m_train['MAE']:.4f} "
          f"| R²={m_train['R2']:.4f} | MAPE={m_train['MAPE']:.2f}%")
    print(f"  │  OOS  → RMSE={m_oos['RMSE']:.4f}   | MAE={m_oos['MAE']:.4f}   "
          f"| R²={m_oos['R2']:.4f}   | MAPE={m_oos['MAPE']:.2f}%")

    all_metrics.append({"horizon_hours": h, "split": "train",
                        **{k: round(v, 6) for k, v in m_train.items()}})
    all_metrics.append({"horizon_hours": h, "split": "oos",
                        **{k: round(v, 6) for k, v in m_oos.items()}})

    # ── 5f. Prediction DataFrames (DSS-standard schema) ──────
    df_pred_train = pd.DataFrame({
        "time"         : idx_train,
        "split"        : "train",
        "horizon_hours": h,
        "actual_hs"    : y_train,
        "predicted_hs" : pred_train,
    })
    df_pred_oos = pd.DataFrame({
        "time"         : idx_oos,
        "split"        : "oos",
        "horizon_hours": h,
        "actual_hs"    : y_oos,
        "predicted_hs" : pred_oos,
    })
    train_pred_frames.append(df_pred_train)
    oos_pred_frames.append(df_pred_oos)

    # ── 5g. Save Model Weights ───────────────────────────────
    model_path = os.path.join(OUTPUT_DIR, f"ebm_model_{h}h.pkl")
    with open(model_path, "wb") as f:
        pickle.dump({"model": ebm, "selector": selector,
                     "selected_features": selected_feats,
                     "horizon_hours": h, "step_shift": step_shift}, f)
    print(f"  └─ Model saved → {model_path}")

# ── 6. EXPORT METRICS & PREDICTIONS ─────────────────────────
print("\n[3/6] Exporting metrics and prediction tables...")

df_metrics = pd.DataFrame(all_metrics)
metrics_path = os.path.join(OUTPUT_DIR, "ebm_metrics_summary.csv")
df_metrics.to_csv(metrics_path, index=False)
print(f"    [✓] Metrics  → {metrics_path}")
print(df_metrics.to_string(index=False))

df_train_all = pd.concat(train_pred_frames, ignore_index=True)
df_oos_all   = pd.concat(oos_pred_frames,   ignore_index=True)

train_pred_path = os.path.join(OUTPUT_DIR, "ebm_train_predictions.csv")
oos_pred_path   = os.path.join(OUTPUT_DIR, "ebm_oos_predictions.csv")

df_train_all.to_csv(train_pred_path, index=False)
df_oos_all.to_csv(oos_pred_path,     index=False)
print(f"    [✓] Train predictions → {train_pred_path}")
print(f"    [✓] OOS   predictions → {oos_pred_path}")

# ── 7. Q1 JOURNAL VISUALISATIONS (6h Horizon, OOS set) ──────
print("\n[4/6] Generating Q1 journal-grade visualisations (+6h, OOS)...")

VIZ_HORIZON  = 6
VIZ_WINDOW_D = 14      # days around peak Hs event
DPI          = 600
FMTS         = ["png", "tiff"]

sns.set_style("ticks")
PLOT_PARAMS = dict(
    font_scale   = 1.25,
    rc           = {"axes.spines.top": False, "axes.spines.right": False,
                    "font.family": "serif"},
)
sns.set(**{k: v for k, v in PLOT_PARAMS.items() if k != "rc"})
plt.rcParams.update(PLOT_PARAMS["rc"])
plt.rcParams["font.size"] = 11

def save_fig(fig, stem: str):
    for fmt in FMTS:
        fpath = os.path.join(OUTPUT_DIR, f"{stem}.{fmt}")
        fig.savefig(fpath, dpi=DPI, bbox_inches="tight", format=fmt)
        print(f"      Saved → {fpath}")
    plt.close(fig)

# Filter 6h OOS predictions
oos_6h = df_oos_all[df_oos_all["horizon_hours"] == VIZ_HORIZON].copy()
oos_6h["time"] = pd.to_datetime(oos_6h["time"])
oos_6h = oos_6h.sort_values("time")

# ── FIG 1: Density Scatter (Actual vs. Predicted) ───────────
try:
    print("\n  [Fig 1] Density scatter — Actual vs. Predicted Hs (+6h OOS)")
    from matplotlib.colors import Normalize
    from scipy.stats import gaussian_kde

    act  = oos_6h["actual_hs"].values
    pred = oos_6h["predicted_hs"].values

    # Kernel density for colour encoding
    xy   = np.vstack([act, pred])
    kde  = gaussian_kde(xy)(xy)
    idx_sort = kde.argsort()

    lim_min = min(act.min(), pred.min()) * 0.95
    lim_max = max(act.max(), pred.max()) * 1.05

    fig, ax = plt.subplots(figsize=(6.5, 6.0))
    sc = ax.scatter(act[idx_sort], pred[idx_sort],
                    c=kde[idx_sort], s=8, cmap="viridis",
                    alpha=0.75, linewidths=0, rasterized=True)
    cbar = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.046)
    cbar.set_label("Kernel Density", fontsize=10)

    # 1:1 identity line
    ax.plot([lim_min, lim_max], [lim_min, lim_max],
            "r--", lw=1.8, label="1:1 Line", zorder=5)

    # Annotate metrics
    m = df_metrics[(df_metrics["horizon_hours"] == VIZ_HORIZON) &
                   (df_metrics["split"] == "oos")].iloc[0]
    txt = (f"RMSE = {m['RMSE']:.3f} m\n"
           f"MAE  = {m['MAE']:.3f} m\n"
           f"R²   = {m['R2']:.4f}\n"
           f"MAPE = {m['MAPE']:.2f}%")
    ax.text(0.04, 0.96, txt, transform=ax.transAxes,
            fontsize=9.5, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.7", alpha=0.9))

    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.set_xlabel("Observed $H_s$ (m)", fontsize=12)
    ax.set_ylabel("Predicted $H_s$ (m)", fontsize=12)
    ax.set_title("EBM: Observed vs. Predicted $H_s$ — +6 h Horizon (OOS)", fontsize=12)
    ax.legend(fontsize=10)
    ax.set_aspect("equal")
    sns.despine(ax=ax)
    fig.tight_layout()
    save_fig(fig, "fig1_ebm_density_scatter_6h")

except Exception as e:
    print(f"    [!] Fig 1 skipped due to error: {e}")

# ── FIG 2: 14-Day Time-Series Window Around Peak Hs ─────────
try:
    print("\n  [Fig 2] 14-day time-series window around peak Hs (+6h OOS)")

    peak_time  = oos_6h.loc[oos_6h["actual_hs"].idxmax(), "time"]
    half_win   = pd.Timedelta(days=VIZ_WINDOW_D / 2)
    t_start    = peak_time - half_win
    t_end      = peak_time + half_win

    win = oos_6h[(oos_6h["time"] >= t_start) & (oos_6h["time"] <= t_end)].copy()
    print(f"      Peak Hs at {peak_time} | Window: {t_start.date()} → {t_end.date()} "
          f"({len(win)} samples)")

    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.plot(win["time"], win["actual_hs"],
            lw=1.5, color="#1f77b4", label="Observed $H_s$", zorder=3)
    ax.plot(win["time"], win["predicted_hs"],
            lw=1.5, color="#d62728", ls="--", label="EBM Predicted $H_s$", zorder=3)
    ax.axvline(peak_time, color="0.4", lw=1.2, ls=":", label=f"Peak ({peak_time.date()})")
    ax.fill_between(win["time"],
                    win["actual_hs"], win["predicted_hs"],
                    alpha=0.12, color="grey", label="Residual")

    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Significant Wave Height $H_s$ (m)", fontsize=12)
    ax.set_title("EBM: 14-Day Event Window — +6 h Horizon (OOS)", fontsize=12)
    ax.legend(fontsize=10, ncol=4, loc="upper left")
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %d\n%Y"))
    sns.despine(ax=ax)
    fig.tight_layout()
    save_fig(fig, "fig2_ebm_timeseries_event_6h")

except Exception as e:
    print(f"    [!] Fig 2 skipped due to error: {e}")

# ── FIG 3: Global Feature Importance (Top 10) ───────────────
try:
    print("\n  [Fig 3] Global EBM feature importance — Top 10 (+6h)")

    ebm_6h       = trained_models[VIZ_HORIZON]["model"]
    feat_6h      = trained_models[VIZ_HORIZON]["features"]

    # Extract global explanation from EBM internals
    global_exp   = ebm_6h.explain_global()
    exp_data     = global_exp.data()

    raw_names    = exp_data["names"]
    raw_scores   = exp_data["scores"]

    # Sort descending, take top 10
    sorted_idx   = np.argsort(raw_scores)[::-1][:10]
    top_names    = [raw_names[i] for i in sorted_idx]
    top_scores   = [raw_scores[i] for i in sorted_idx]

    # Reverse for horizontal bar chart (highest at top)
    top_names  = top_names[::-1]
    top_scores = top_scores[::-1]

    palette = sns.color_palette("Blues_d", len(top_names))

    fig, ax = plt.subplots(figsize=(8.0, 5.5))
    bars = ax.barh(top_names, top_scores, color=palette, edgecolor="0.3", linewidth=0.5)

    # Value labels
    for bar, val in zip(bars, top_scores):
        ax.text(bar.get_width() + max(top_scores) * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", ha="left", fontsize=9)

    ax.set_xlabel("Mean Absolute Score (EBM Importance)", fontsize=12)
    ax.set_title("EBM Global Feature Importance — Top 10 (+6 h Horizon)", fontsize=12)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    sns.despine(ax=ax)
    fig.tight_layout()
    save_fig(fig, "fig3_ebm_global_importance_6h")

except Exception as e:
    print(f"    [!] Fig 3 skipped due to error: {e}")

# ── 8. FINAL SUMMARY ─────────────────────────────────────────
print("\n[5/6] Pipeline complete. Artefact inventory:")
print("-" * 65)
artefacts = sorted(os.listdir(OUTPUT_DIR))
for a in artefacts:
    fpath = os.path.join(OUTPUT_DIR, a)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"    {a:<50} {size_kb:>8.1f} KB")

print("\n[6/6] Multi-horizon EBM training pipeline finished successfully.")
print("=" * 65)
print("  Horizons trained  : ", HORIZONS)
print("  Top features/model: ", TOP_K)
print("  Output directory  : ", OUTPUT_DIR)
print("=" * 65)

   PHASE 5 — EBM MULTI-HORIZON TRAINING PIPELINE
Mounted at /content/drive

[✓] Output directory ready: /content/drive/MyDrive/KBS_Paper/Outputs/5_EBM_KBS/

[1/6] Loading feature-engineered datasets...
    Train shape : (12477, 722)
    OOS   shape : (5605, 722)
    Train period: 2018-07-10 06:00:00 → 2022-10-17 06:00:00
    OOS   period: 2022-10-18 09:00:00 → 2024-09-18 06:00:00

[2/6] Starting multi-horizon EBM training loop...
-----------------------------------------------------------------

  ┌─ Horizon: +3h  (step_shift = 1 × 3h)
  │  Train samples: 12,476  |  OOS samples: 5,604
  │  Feature pool : 721 features
  │  Running SelectKBest (k=100) to reduce RAM footprint...
  │  Selected features: 100
  │  Initialising ExplainableBoostingRegressor...
  │  Fitting EBM on 12,476 training samples...
  │  Training complete.
  │  Train → RMSE=0.1792 | MAE=0.0770 | R²=0.5977 | MAPE=20.82%
  │  OOS  → RMSE=0.2245   | MAE=0.0836   | R²=0.4480   | MAPE=20.54%
  └─ Model saved → /content/drive